<a href="https://colab.research.google.com/github/mofidumar15/AGENTIC-RAG-COURSE-PLANNING-ASSISTANT/blob/main/AGENTIC_RAG_COURSE_PLANNING_ASSISTANT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y langgraph-prebuilt langgraph-checkpoint langgraph -q

In [ ]:
!pip install -q \
langchain==0.2.14 \
langchain-community==0.2.12 \
langchain-core==0.2.33 \
langchain-huggingface \
chromadb==0.5.5 \
sentence-transformers==2.6.1 \
transformers==4.41.2 \
numpy==1.26.4 \
accelerate \
bitsandbytes \
torch

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

import os

In [ ]:
urls = [
    "https://catalog.illinois.edu/courses-of-instruction/cs/?utm_source=chatgpt.com",
    "https://www2.eecs.berkeley.edu/Courses/CS/?utm_source=chatgpt.com",
    "https://catalog.registrar.uiowa.edu/courses/cs/?utm_source=chatgpt.com",
    "https://catalog.ucsd.edu/courses/CSE.html?utm_source=chatgpt.com",
    "https://catalog.uoregon.edu/courses/crs-cs/?utm_source=chatgpt.com",
    "https://cse.engin.umich.edu/academics/course-resources/cse-course-info/?utm_source=chatgpt.com",
    "https://www.bc.edu/bc-web/academics/sites/university-catalog/courses/arts-sciences/computer-science.html?utm_source=chatgpt.com",
    "https://catalog.uab.edu/coursedescriptions/cs/?utm_source=chatgpt.com",
    "https://catalog.colostate.edu/general-catalog/courses-az/cs/?utm_source=chatgpt.com",
    "https://catalog.utexas.edu/general-information/coursesatoz/c-s/?utm_source=chatgpt.com",
    "https://catalog.purdue.edu/content.php?catoid=15&navoid=18621",
    "https://catalog.purdue.edu/preview_course_nopop.php?catoid=15&coid=230870",
    "https://catalog.purdue.edu/preview_course_nopop.php?catoid=15&coid=230871",
    "https://catalog.purdue.edu/preview_program.php?catoid=15&poid=19990",
    "https://bulletin.columbia.edu/columbia-college/departments-instruction/computer-science/",
    "https://catalog.mit.edu/subjects/6/",
    "https://catalog.gatech.edu/coursesaz/cs/",
    "https://catalog.umd.edu/courses/cmsc/",
    "https://catalog.utexas.edu/undergraduate/natural-sciences/degrees-and-programs/bs-computer-science/",
    "https://catalog.ucsd.edu/curric/CSE-ug.html",
    "https://catalog.illinois.edu/undergraduate/engineering/computer-science-bs/",
    "https://catalog.illinois.edu/academic-policies/",
    "https://catalog.utexas.edu/general-information/academic-policies/",
    "https://catalog.ucsd.edu/academic-policies/index.html",
    "https://bulletin.columbia.edu/columbia-college/regulations/",

]

In [ ]:
all_docs = []

for url in urls:
    try:
        loader = WebBaseLoader(url)
        docs = loader.load()
        all_docs.extend(docs)
    except Exception as e:
        print(f"Error loading {url}: {e}")

print(f"Loaded {len(all_docs)} documents")

Loaded 25 documents


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

split_docs = text_splitter.split_documents(all_docs)
print(f"Total chunks: {len(split_docs)}")

Total chunks: 2447


In [ ]:

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
print(type(vectorstore))

<class 'langchain_community.vectorstores.chroma.Chroma'>


In [ ]:
prompt_template = """
You are a strict academic assistant.

RULES:
- ONLY answer using provided context
- If missing info, say: "I don’t have that information in the provided catalog"
- DO NOT guess
- ALWAYS include citations

OUTPUT FORMAT:

Answer / Plan:
Why (requirements/prereqs satisfied):
Citations:
Clarifying questions:
Assumptions / Not in catalog:

Context:
{context}

Question:
{question}
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

In [ ]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline


pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_length=512,
    temperature=0
)

llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)

In [ ]:
def agentic_planner(student_profile, question):
    missing = []

    if "major" not in student_profile:
        missing.append("What is your major?")

    if "completed_courses" not in student_profile:
        missing.append("Which courses have you completed?")

    if missing:
        return {
            "Clarifying questions": missing
        }
    response = qa_chain({"query": question})

    return response

In [ ]:
student = {
    "major": "Computer Science",
    "completed_courses": ["CS101", "CS102", "MATH120"],
    "target_term": "Spring",
    "interests": ["AI", "Databases"],
    "credits_completed": 15,
    "transfer_student": False
}

question = """
I want a recommended course plan for next semester.
Check prerequisites strictly and suggest only eligible courses.
Also mention risks or missing requirements.
"""

response = agentic_planner(student, question)
print(response)

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


{'query': '\nI want a recommended course plan for next semester.\nCheck prerequisites strictly and suggest only eligible courses.\nAlso mention risks or missing requirements.\n', 'result': 'RULES: - ONLY answer using provided context - DO NOT guess - ALWAYS include citations - If missing info, say: "I don’t have that information in the provided catalog" - DO NOT guess', 'source_documents': [Document(metadata={'language': 'en', 'source': 'https://catalog.utexas.edu/general-information/coursesatoz/c-s/?utm_source=chatgpt.com', 'title': 'C S - Computer Science < The University of Texas at Austin'}, page_content='Courses'), Document(metadata={'language': 'en', 'source': 'https://catalog.utexas.edu/undergraduate/natural-sciences/degrees-and-programs/bs-computer-science/', 'title': 'Bachelor of Science in Computer Science < The University of Texas at Austin'}, page_content='Courses'), Document(metadata={'language': 'en', 'source': 'https://catalog.utexas.edu/general-information/coursesatoz/c

In [ ]:
test_queries = [

# PREREQUISITE
"Can I take CS301 after completing CS101?",
"What do I need before taking Database Systems?",
"Is CS201 required before CS301?",
"Can I enroll in AI course without MATH120?",
"Do I need programming experience for CS101?",
"Is CS102 a prerequisite for CS202?",
"Can I take Operating Systems after Data Structures?",
"What courses are required before Machine Learning?",
"Is calculus required for Computer Science major?",
"Do I need CS101 before taking CS105?",
"Can I skip CS201 if I have equivalent experience?",
"Is there a minimum grade required in CS101 to take CS201?",
"Can I take CS301 if I passed CS201 with C grade?",
"Is MATH101 required before CS102?",
"Can I take advanced CS courses without introductory courses?",

# PREREQUISITE CHAINS
"What is the full prerequisite chain for Machine Learning?",
"What courses must I complete before taking Advanced Algorithms?",
"If I completed CS101, what is the path to AI course?",
"What sequence of courses leads to Database Systems?",
"Explain prerequisite path from CS101 to CS401",
"What do I need before taking Deep Learning?",
"What are all requirements before taking capstone project?",
"How do I progress from beginner to advanced CS courses?",
"What courses must be taken before Distributed Systems?",
"List all dependencies before enrolling in Operating Systems",

# PROGRAM REQUIREMENTS

"How many credits are required for a CS degree?",
"What are the core courses required for Computer Science major?",
"Are electives required in the CS program?",
"What categories of courses must I complete to graduate?",
"Is there a residency requirement for the degree?",
"What GPA is required to graduate?",
"How many elective credits are needed?",
"Are internships counted toward degree credits?",
"Can I take courses from other departments as electives?",
"What are the graduation requirements for CS program?",


# TRICKS / NOT IN DOC
"Is CS301 offered in Fall semester?",
"Who is the professor for Database Systems?",
"What time is CS101 scheduled?",
"How difficult is the AI course?",
"Which course is the easiest in CS department?"
]

In [ ]:
for q in test_queries:
    print(f"\nQUESTION: {q}")
    res = qa_chain.invoke({"query": q})
    print(res["result"])


QUESTION: Can I take CS301 after completing CS101?


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


No

QUESTION: What do I need before taking Database Systems?


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Computer Science 313E, 314, or 314H with a grade of at least C-

QUESTION: Is CS201 required before CS301?


...

CS101 is scheduled at 3:30 PM

QUESTION: How difficult is the AI course?


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


a high-level understanding of the AI field

QUESTION: Which course is the easiest in CS department?


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


CS:1020 Principles of Computing


INTELLIGENT STUDENT INTAKE AGENT

In [ ]:
def intake_agent(student_profile):
    missing = []

    required_fields = ["major", "completed_courses", "target_term"]

    for field in required_fields:
        if field not in student_profile:
            missing.append(field)

    if missing:
        return {
            "status": "NEED_INFO",
            "missing_fields": missing,
            "message": "Please provide missing student information."
        }

    return {
        "status": "OK",
        "student_profile": student_profile
    }

RETRIEVER AGENT

In [ ]:
def retriever_agent(question):
    docs = retriever.invoke(question)

    context = "\n\n".join([
        d.page_content for d in docs
    ])

    sources = [d.metadata.get("source", "Unknown") for d in docs]

    return {
        "context": context,
        "sources": sources
    }

In [ ]:
def planner_agent(student_profile, question, context):

    enriched_prompt = f"""
You are an academic advisor assistant.

RULES:
- Use ONLY the provided context
- If info missing → say "Not in catalog"
- MUST give structured output
- MUST check prerequisites carefully

Student Profile:
Major: {student_profile['major']}
Completed Courses: {student_profile['completed_courses']}
Target Term: {student_profile['target_term']}

Context:
{context}

Question:
{question}

FORMAT:
Answer:
Decision (Eligible / Not Eligible / Need More Info):
Why:
Next Step:
"""

    response = llm.invoke(enriched_prompt)

    return response

In [ ]:
def verifier_agent(response, sources):

    prompt = f"""
You are a strict verifier.

Check:
- Is answer based ONLY on sources?
- Are there unsupported claims?
- Is refusal needed?

Sources:
{sources}

Answer:
{response}

Return:
- VALID or INVALID
- If invalid, explain why
"""

    return llm.invoke(prompt)

In [ ]:
def agentic_system(student_profile, question):


    intake = intake_agent(student_profile)

    if intake["status"] == "NEED_INFO":
        return intake


    retrieval = retriever_agent(question)


    plan = planner_agent(
        student_profile,
        question,
        retrieval["context"]
    )


    verification = verifier_agent(plan, retrieval["sources"])

    return {
        "FINAL_ANSWER": plan,
        "VERIFICATION": verification,
        "SOURCES": retrieval["sources"]
    }

In [ ]:
student = {
    "major": "Computer Science",
    "completed_courses": ["CS101", "CS102", "MATH120"],
    "target_term": "Fall"
}

question = "Can I take Database Systems after CS101?"

result = agentic_system(student, question)

print("\n===== FINAL ANSWER =====\n")
print(result["FINAL_ANSWER"])

print("\n===== VERIFICATION =====\n")
print(result["VERIFICATION"])

print("\n===== SOURCES =====\n")
print(result["SOURCES"])

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:141: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  warn_deprecated(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(



===== FINAL ANSWER =====

CS4400. Introduction to Database Systems. 3 Credit Hours. Comprehensive coverage of mainstream database concepts such as the entity-relationship model, relational databases, query languages, and database design methodology. Includes a project. Credit not allowed for both CS 4400 and CS 6402. CS 4420. Database System Implementation. 3 Credit Hours. Comprehensive coverage of mainstream database concepts such as the entity-relationship model, relational databases, query languages, and database design methodology. Includes a project. Credit not allowed for both CS 4420 and CS 6402. CS 4420. Database System Implementation. 3 Credit Hours. Comprehensive coverage of mainstream database concepts such as the entity-relationship model, relational databases, query languages, and database design methodology. Includes a project. Credit not allowed for both CS 4420 and CS 6402. CS 4420. Database System Implementation. 3 Credit Hours. Comprehensive coverage of mainstream da

In [ ]:
def ask_question(query):
    result = qa_chain.invoke({"query": query})

    print("\n============================")
    print("📌 QUESTION:")
    print(query)

    print("\n✅ ANSWER:")
    print(result["result"])

    print("\n📚 SOURCES:")
    for doc in result["source_documents"]:
        print("-", doc.metadata.get("source", "Unknown"))



In [ ]:
while True:
    user_input = input("\n🎓 Ask your course planning question (or type 'exit'): ")

    if user_input.lower() == "exit":
        print("👋 Exiting system...")
        break

    ask_question(user_input)


🎓 Ask your course planning question (or type 'exit'): can i take CS101


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(



📌 QUESTION:
can i take CS101

✅ ANSWER:
No

📚 SOURCES:
- https://catalog.colostate.edu/general-catalog/courses-az/cs/?utm_source=chatgpt.com
- https://catalog.colostate.edu/general-catalog/courses-az/cs/?utm_source=chatgpt.com
- https://catalog.colostate.edu/general-catalog/courses-az/cs/?utm_source=chatgpt.com
- https://catalog.colostate.edu/general-catalog/courses-az/cs/?utm_source=chatgpt.com
- https://catalog.colostate.edu/general-catalog/courses-az/cs/?utm_source=chatgpt.com

🎓 Ask your course planning question (or type 'exit'): Who is the professor for Database Systems


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(



📌 QUESTION:
Who is the professor for Database Systems

✅ ANSWER:
C S 386D

📚 SOURCES:
- https://catalog.utexas.edu/general-information/coursesatoz/c-s/?utm_source=chatgpt.com
- https://catalog.utexas.edu/general-information/coursesatoz/c-s/?utm_source=chatgpt.com
- https://catalog.ucsd.edu/courses/CSE.html?utm_source=chatgpt.com
- https://catalog.ucsd.edu/courses/CSE.html?utm_source=chatgpt.com
- https://catalog.utexas.edu/general-information/coursesatoz/c-s/?utm_source=chatgpt.com

🎓 Ask your course planning question (or type 'exit'): EXIT
👋 Exiting system...
